# Drogued drifter in CMEMS Baltic currents

Proof of concept coupling the `DroguedDrifter` model to real CMEMS Baltic Sea currents via Parcels v4. This combines the CMEMS pipeline from notebook 02 with the drogued drifter kernel from notebook 01.

The drifter kernel passes a `sample_uv(z)` closure to the drifter model, which queries the FieldSet at arbitrary depth during ODE integration. We compare drogued drifter trajectories against surface point particles advected with AdvectionRK4.

This is a pipeline proof of concept, not a validation against observations.

## Imports

In [ ]:
import shutil
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from parcels import FieldSet, Particle, ParticleFile, ParticleSet, StatusCode
from parcels.kernels import AdvectionEE

from drogued_drifters.drifter import DroguedDrifter, make_dd_velocity_interpolator

## Parameters

In [ ]:
N_PARTICLES = 20
LON_MIN = 10.0
LON_MAX = 11.0
LAT_MIN = 54.3
LAT_MAX = 54.6
RANDOM_SEED = 42
DROGUE_DEPTH = 3.0
DT = 300.0
RUNTIME = 172800.0  # 48 hours [s]
OUTPUTDT = 3600.0
CMEMS_DIR = "data/cmems"

## Load CMEMS data

Same pipeline as notebook 02: open the CMEMS Baltic Sea physics product lazily, subset to the Kiel Bight, rename velocity components, fill land with zero, and add SGRID topology for Parcels.

In [ ]:
ds_sub = xr.open_dataset(CMEMS_DIR + "/cmems_mod_bal_phy_anfc_PT1H-i.nc").sel(
    longitude=slice(9.5, 11.5),
    latitude=slice(54.0, 55.5),
    time=slice("2023-04-24", "2023-04-27"),
)

ds_sub = ds_sub.rename({"uo": "U", "vo": "V"}).fillna(0.0)

# Add z=0 layer by copying shallowest level (the CMEMS top cell is a
# near-surface average). Without this, sampling at z=0 is out of bounds.
ds_z0 = ds_sub.isel(depth=0).assign_coords(depth=0.0)
ds_sub = xr.concat([ds_z0, ds_sub], dim="depth")

ds_sub = ds_sub.load()
ds_sub

In [ ]:
ds_sub["grid"] = xr.DataArray(
    data=0,
    attrs={
        "cf_role": "grid_topology",
        "topology_dimension": 2,
        "node_dimensions": "longitude latitude",
        "face_dimensions": (
            "longitude:longitude (padding: none) "
            "latitude:latitude (padding: none)"
        ),
        "vertical_dimensions": "depth:depth (padding: none)",
        "node_coordinates": "longitude latitude",
    },
)

fieldset = FieldSet.from_sgrid_conventions(ds_sub, mesh="spherical")

## Drogued drifter interpolator

Replace the default Parcels velocity interpolator with `make_dd_velocity_interpolator`.
At each RHS evaluation, the interpolator extracts the full velocity profile at each
particle position (bilinear in t, y, x at every depth level), builds a fast
`sample_uv(z)` closure, and runs `DroguedDrifter.get_final_drift_batch` to obtain the
steady-state drift velocity. `AdvectionEE` then steps particles with that drift.

In [ ]:
dd = DroguedDrifter()
fieldset.UV.vector_interp_method = make_dd_velocity_interpolator(dd, spherical=True)

# Point particle runs need a separate FieldSet with the default interpolator.
fieldset_pp = FieldSet.from_sgrid_conventions(ds_sub, mesh="spherical")


def DeleteOOB(particles, fieldset):
    state = np.asarray(particles.state)
    oob = (state == StatusCode.ErrorOutOfBounds) | (state == StatusCode.ErrorThroughSurface)
    if np.any(oob):
        particles.state = np.where(oob, StatusCode.Delete, state)

## Release particles

20 random positions in the southern Kiel Bight.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
release_lons = rng.uniform(LON_MIN, LON_MAX, N_PARTICLES)
release_lats = rng.uniform(LAT_MIN, LAT_MAX, N_PARTICLES)

## Run simulations

Three runs: drogued drifter (`AdvectionEE` with the DD interpolator), surface point
particles (`AdvectionEE` at z=0), and 3 m point particles (`AdvectionEE` at drogue
depth). All output to zarr.

In [ ]:
SHALLOWEST_DEPTH = float(ds_sub.depth.values[0])
OUTPUTDT = 3600.0

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Drogued drifter run — uses the custom DD velocity interpolator
dd_store = str(OUTPUT_DIR / "cmems_drogued_drifter.zarr")
shutil.rmtree(dd_store, ignore_errors=True)

pset_drifter = ParticleSet(
    fieldset=fieldset,
    pclass=Particle,
    lon=release_lons.tolist(),
    lat=release_lats.tolist(),
    z=[SHALLOWEST_DEPTH] * N_PARTICLES,
)
pset_drifter.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=dd_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

In [ ]:
# Surface point particle run
surface_store = str(OUTPUT_DIR / "cmems_surface.zarr")
shutil.rmtree(surface_store, ignore_errors=True)

pset_surface = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons.tolist(),
    lat=release_lats.tolist(),
    z=[SHALLOWEST_DEPTH] * N_PARTICLES,
)
pset_surface.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=surface_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

In [ ]:
# 3 m point particle run
DROGUE_DEPTH_LEVEL = float(ds_sub.depth.sel(depth=3.0, method="nearest"))

drogue_store = str(OUTPUT_DIR / "cmems_drogue.zarr")
shutil.rmtree(drogue_store, ignore_errors=True)

pset_drogue = ParticleSet(
    fieldset=fieldset_pp,
    pclass=Particle,
    lon=release_lons.tolist(),
    lat=release_lats.tolist(),
    z=[DROGUE_DEPTH_LEVEL] * N_PARTICLES,
)
pset_drogue.execute(
    kernels=[AdvectionEE, DeleteOOB],
    dt=DT,
    runtime=RUNTIME,
    output_file=ParticleFile(store=drogue_store, outputdt=OUTPUTDT),
    verbose_progress=False,
)

## Plot trajectories

Surface point particles (red), 3m point particles (green), drogued drifters (blue, dashed).

In [ ]:
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

ds_drifter = xr.open_zarr(dd_store)
ds_surface = xr.open_zarr(surface_store)
ds_drogue = xr.open_zarr(drogue_store)

for i in range(ds_surface.sizes["trajectory"]):
    valid = np.isfinite(ds_surface.lon.values[i])
    ax.plot(ds_surface.lon.values[i, valid], ds_surface.lat.values[i, valid],
            color="tab:red", label="Surface point particle" if i == 0 else None)

for i in range(ds_drogue.sizes["trajectory"]):
    valid = np.isfinite(ds_drogue.lon.values[i])
    ax.plot(ds_drogue.lon.values[i, valid], ds_drogue.lat.values[i, valid],
            color="tab:green", label="3 m point particle" if i == 0 else None)

for i in range(ds_drifter.sizes["trajectory"]):
    valid = np.isfinite(ds_drifter.lon.values[i])
    ax.plot(ds_drifter.lon.values[i, valid], ds_drifter.lat.values[i, valid],
            color="tab:blue", ls="--", label="Drogued drifter" if i == 0 else None)

land = cfeature.NaturalEarthFeature("physical", "land", "10m",
                                     facecolor="0.9", edgecolor="0.5")
ax.add_feature(land)
ax.set_extent([9.5, 11.5, 53.8, 55.5])
ax.gridlines(draw_labels=True)
ax.legend()
plt.show()